# Getting started

This tutorial walks through the core workflow of `molisanax`:

1. Build a synthetic forcing field of two counter-rotating geostrophic eddies and wrap it in a `Dataset`.
2. Run a **deterministic** trajectory through the field with the `solve` ODE integrator.
3. Run a **stochastic** ensemble using a Smagorinsky-style diffusion term — exercising the SDE mode of `solve` and the `Dataset.neighborhood` API.
4. **Learn** parameters of a tunable term by JAX automatic differentiation, using `optimistix` and the `liu_index` metric as a loss.

The same pattern transposes to real forcing fields loaded from xarray (zarr / netCDF) via `Dataset.from_xarray`.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import numpy as np
import matplotlib.pyplot as plt


## 1. Defining a velocity field

We build an idealised sea surface height (SSH) field with two Gaussian bumps of opposite sign, slowly drifting apart over two days. From SSH we derive a geostrophic surface velocity field $(u, v)$ via

$$
u = -\frac{g}{f} \, \partial_y \eta, \qquad v = \frac{g}{f} \, \partial_x \eta,
$$

where $f$ is the Coriolis parameter at the mean latitude.

In [ ]:
from molisanax import degrees_to_meters

ny = nx = 101
nt = 49

lat = jnp.linspace(29.0, 31.0, ny)
lon = jnp.linspace(-1.0, 1.0, nx)

dt = 60 * 60  # one hour, in seconds
ts = jnp.linspace(0.0, dt * (nt - 1), nt)  # 48 hours, hourly

dy, dx = degrees_to_meters(jnp.asarray([lat[1] - lat[0], lon[1] - lon[0]]), lat.mean())


In [ ]:
# SSH field: two Gaussian bumps drifting apart
eta0 = 0.4
sig = 0.1
def eta_fn(_dx, _dy):
    return eta0 * jnp.exp(-((_dx / (sig * nx)) ** 2) - (_dy / (sig * ny)) ** 2)

xi, yi = jnp.meshgrid(jnp.arange(lon.size), jnp.arange(lat.size), indexing="ij")

eddy_speed = 0.05  # m/s
dxi = eddy_speed * dt / dx
dyi = eddy_speed * dt / dy

def ssh_at(t):
    x1 = 0.5 * lon.size - (8 + dxi * t)
    y1 = 0.5 * lat.size - (8 + dyi * t)
    x2 = 0.5 * lon.size + (8 + dxi * t)
    y2 = 0.5 * lat.size + (8 + dyi * t)
    return eta_fn(x1 - xi, y1 - yi) - eta_fn(x2 - xi, y2 - yi)

eta = jax.vmap(ssh_at)(jnp.arange(nt))  # (nt, nx, ny) — note (xi, yi) order

# Geostrophic velocities (meters / second)
coriolis = 2 * 7.292115e-5 * jnp.sin(jnp.radians(lat.mean()))
g = 9.81

# eta is (nt, nx, ny); we want u, v on (nt, ny, nx) ordered (time, lat, lon).
eta_tll = eta.transpose(0, 2, 1)  # (nt, ny, nx)

u = jnp.zeros_like(eta_tll)
u = u.at[:, :-1, :].set(jnp.diff(eta_tll, axis=1) / dy / coriolis * g)
u = u.at[:, -1, :].set(u[:, -2, :])

v = jnp.zeros_like(eta_tll)
v = v.at[:, :, :-1].set(-jnp.diff(eta_tll, axis=-1) / dx / coriolis * g)
v = v.at[:, :, -1].set(v[:, :, -2])

print("u, v shape:", u.shape, "  speed range:",
      float(jnp.sqrt(u**2 + v**2).min()), float(jnp.sqrt(u**2 + v**2).max()))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), constrained_layout=True)
for ax, k in zip(axes, [0, nt // 2, nt - 1]):
    speed = jnp.sqrt(u[k] ** 2 + v[k] ** 2)
    im = ax.pcolormesh(lon, lat, speed, vmin=0, vmax=2.5, cmap="viridis")
    ax.quiver(lon[::6], lat[::6], u[k][::6, ::6], v[k][::6, ::6], scale=40, color="white", alpha=0.7)
    ax.set_aspect("equal")
    ax.set_title(f"$t$ = {ts[k] / 3600:.0f} h")
fig.colorbar(im, ax=axes, label="$\\| \\mathbf{u} \\|$ (m s$^{-1}$)", shrink=0.8)
plt.show()


Now wrap the velocity arrays in a `Dataset` so they can be queried by the integrator. `Dataset.from_arrays` accepts plain numpy or JAX arrays directly; for real datasets, use `Dataset.from_xarray` instead.

In [ ]:
from molisanax import Dataset

two_eddies = Dataset.from_arrays(
    fields={"u": u, "v": v},
    t=ts,
    lat=lat,
    lon=lon,
    dtype=jnp.float64,
)


## 2. Simulating trajectories

### 2.1 A single deterministic trajectory

A particle is advected by the surface current:

$$
\mathrm{d}\mathbf{X}(t) = \mathbf{u}(t, \mathbf{X}(t)) \, \mathrm{d}t.
$$

In `molisanax` the dynamics are expressed as a Python callable `term(t, y, args)` that returns the velocity in degrees per second. `solve` defaults to ODE mode and uses the `Heun` integrator with a fixed step size.

In [ ]:
from molisanax import solve, Heun, meters_to_degrees

def uv_term(t, y, args):
    dataset = args
    lat_, lon_ = y[0], y[1]
    u_ = dataset["u"].interp(t, lat_, lon_)  # m/s
    v_ = dataset["v"].interp(t, lat_, lon_)  # m/s
    # convert (north, east) m/s -> (dlat/dt, dlon/dt) deg/s
    return meters_to_degrees(jnp.array([v_, u_]), lat_)

y0 = jnp.array([30.0, 0.0])           # initial [lat, lon]
ts_sim = ts[1:-1]                     # avoid the boundary timestamps

traj = solve(uv_term, two_eddies, y0, ts_sim, Heun())
print("trajectory shape:", traj.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
speed_max = jnp.sqrt(u ** 2 + v ** 2).max(axis=0)
im = ax.pcolormesh(lon, lat, speed_max, cmap="viridis", vmin=0)
fig.colorbar(im, ax=ax, label="max $\\| \\mathbf{u} \\|$ (m s$^{-1}$)")
ax.plot(traj[:, 1], traj[:, 0], color="tab:orange", lw=2, label="trajectory")
ax.scatter(traj[0, 1], traj[0, 0], color="white", edgecolor="black", zorder=3, label="start")
ax.set_aspect("equal")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.legend(loc="upper right")
plt.show()


### 2.2 A stochastic ensemble

We now switch to SDE mode by passing `key` and `n_noise` to `solve`. The term receives a pre-sampled noise vector $\mathbf{z}$ as a fourth argument and returns the **full velocity** (drift + noise term). A single step is therefore $\mathrm{d}\mathbf{X} = \text{term}(t, \mathbf{X}, \mathrm{args}, \mathbf{z}) \, \mathrm{d}t$.

For the diffusion model we use a Smagorinsky-style local turbulent viscosity. The eddy diffusivity at a point is

$$
K = (C_S \, \Delta x)^2 \sqrt{2\left(\partial_x u\right)^2 + 2\left(\partial_y v\right)^2 + \left(\partial_y u + \partial_x v\right)^2},
$$

estimated from the central differences of $(u, v)$ over a $3 \times 3$ patch obtained via `Dataset.neighborhood`. The noise amplitude is then $\sqrt{2 K / \Delta t}$ (so that the per-step displacement variance is $2 K \, \Delta t$).

In [ ]:
from molisanax import safe_sqrt

CS = 0.2

# spatial step in metres (constant grid)
dy_m, dx_m = degrees_to_meters(
    jnp.asarray([lat[1] - lat[0], lon[1] - lon[0]]), lat.mean()
)
dt_step = float(ts_sim[1] - ts_sim[0])

def smag_term(t, y, args, z):
    dataset = args
    lat_, lon_ = y[0], y[1]

    # advective drift
    u_ = dataset["u"].interp(t, lat_, lon_)
    v_ = dataset["v"].interp(t, lat_, lon_)
    drift = meters_to_degrees(jnp.array([v_, u_]), lat_)

    # 3x3 spatial neighbourhoods centred on the nearest grid point
    patches = dataset.neighborhood(t, lat_, lon_, t_window=0, lat_window=1, lon_window=1)
    u_p = patches["u"][0]  # (3, 3) (lat, lon)
    v_p = patches["v"][0]

    # central differences -> deformation tensor components
    du_dx = (u_p[1, 2] - u_p[1, 0]) / (2 * dx_m)
    du_dy = (u_p[2, 1] - u_p[0, 1]) / (2 * dy_m)
    dv_dx = (v_p[1, 2] - v_p[1, 0]) / (2 * dx_m)
    dv_dy = (v_p[2, 1] - v_p[0, 1]) / (2 * dy_m)

    strain = safe_sqrt(2 * du_dx ** 2 + 2 * dv_dy ** 2 + (du_dy + dv_dx) ** 2)
    K = (CS * dx_m) ** 2 * strain                           # m^2 / s
    sigma = safe_sqrt(2 * K / dt_step)                      # m / s

    # convert noise amplitude from m/s to deg/s and add to drift
    noise_deg = meters_to_degrees(sigma * z, lat_)
    return drift + noise_deg

key = jr.key(0)
ensemble = solve(
    smag_term, two_eddies, y0, ts_sim,
    key=key, n_noise=2, n_samples=100,
)
print("ensemble shape:", ensemble.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
im = ax.pcolormesh(lon, lat, speed_max, cmap="viridis", vmin=0)
fig.colorbar(im, ax=ax, label="max $\\| \\mathbf{u} \\|$ (m s$^{-1}$)")

for i in range(ensemble.shape[0]):
    ax.plot(ensemble[i, :, 1], ensemble[i, :, 0], color="black", alpha=0.12, lw=0.6)
ax.plot(traj[:, 1], traj[:, 0], color="tab:orange", lw=2, label="deterministic")
ax.scatter(y0[1], y0[0], color="white", edgecolor="black", zorder=3, label="start")

ax.set_aspect("equal")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.legend(loc="upper right")
plt.show()


## 3. Learning a simulator parameterisation

`molisanax`'s `solve` is fully differentiable. We exploit that to **fit** the parameters of a term function: given a reference trajectory $\mathbf{Y}^*$ produced by the *true* dynamics, recover the parameters of a *biased* dynamics model that match it.

**Setup.** Take the velocity field $(u, v)$ from §1 and define a transformed field $(u', v') = ((u - b)/a,\ (v - b)/a)$ with $a = 2$, $b = 0.1$. The *true* term advects with $(u, v)$; the *tunable* term is parameterised as

$$
\text{term}_\theta(t, y) = a_\theta \, \mathbf{u}'(t, y) + b_\theta,
$$

and we want to recover $(a_\theta, b_\theta) \to (a, b)$ purely from the discrepancy of the simulated trajectories.

In [ ]:
a_true, b_true = 2.0, 0.1
trans_u = (u - b_true) / a_true
trans_v = (v - b_true) / a_true
trans_grid = Dataset.from_arrays(
    fields={"u": trans_u, "v": trans_v},
    t=ts, lat=lat, lon=lon,
    dtype=jnp.float64,
)


In [ ]:
class TunableUV(eqx.Module):
    intercept: jax.Array
    slope: jax.Array

    def __call__(self, t, y, args):
        dataset = args
        lat_, lon_ = y[0], y[1]
        u_ = self.slope * dataset["u"].interp(t, lat_, lon_) + self.intercept
        v_ = self.slope * dataset["v"].interp(t, lat_, lon_) + self.intercept
        return meters_to_degrees(jnp.array([v_, u_]), lat_)

term_init = TunableUV(intercept=jnp.array(0.0), slope=jnp.array(1.0))
print(f"initial parameters: intercept={float(term_init.intercept):.3f}, "
      f"slope={float(term_init.slope):.3f}")


In [ ]:
y0_fit = jnp.array([29.95, -0.05])

# Reference trajectory: true dynamics on the un-transformed grid
true_traj = solve(uv_term, two_eddies, y0_fit, ts_sim, Heun(), adjoint="forward")

# Initial estimate: tunable term on the transformed grid, before fitting
init_traj = solve(term_init, trans_grid, y0_fit, ts_sim, Heun(), adjoint="forward")
print("ref shape:", true_traj.shape, "  init shape:", init_traj.shape)


In [ ]:
from molisanax import liu_index
import optimistix as optx

@eqx.filter_jit
def residual_fn(term_module, ref_traj):
    est_traj = solve(term_module, trans_grid, y0_fit, ts_sim, Heun(), adjoint="forward")
    return liu_index(est_traj, ref_traj)

solver = optx.BestSoFarLeastSquares(optx.GaussNewton(rtol=1e-8, atol=1e-8))
sol = optx.least_squares(residual_fn, solver=solver, y0=term_init, args=true_traj)
term_fit = sol.value
print(f"converged in {int(sol.stats['num_steps'])} steps  ->  "
      f"intercept={float(term_fit.intercept):.4f}, slope={float(term_fit.slope):.4f}")
print(f"truth                          ->  intercept={b_true:.4f}, slope={a_true:.4f}")


In [ ]:
final_traj = solve(term_fit, trans_grid, y0_fit, ts_sim, Heun(), adjoint="forward")

fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
im = ax.pcolormesh(lon, lat, speed_max, cmap="viridis", vmin=0)
fig.colorbar(im, ax=ax, label="max $\\| \\mathbf{u} \\|$ (m s$^{-1}$)")

ax.plot(true_traj[:, 1], true_traj[:, 0], color="tab:blue",   lw=2, label="truth")
ax.plot(init_traj[:, 1], init_traj[:, 0], color="tab:red",    lw=1.5, ls="--", label="initial guess")
ax.plot(final_traj[:, 1], final_traj[:, 0], color="tab:orange", lw=2, ls=":", label="fitted")

ax.set_aspect("equal")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.legend(loc="upper right")
plt.show()


## Where to next

- See the [API reference](api.md) for the full surface of `solve`, `Dataset`, and the metric / interpolation helpers.
- Replace `Dataset.from_arrays` with `Dataset.from_xarray` to drive the simulator from real zarr or netCDF forcing fields.
- The `solve` integrator supports both reverse-mode (`jax.grad`) and forward-mode (`jax.jvp`) auto-differentiation; this notebook uses forward mode (`adjoint="forward"`) because `optimistix.GaussNewton` requires a Jacobian.